# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huydang2006/flyrank-ML-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** one row = one `(content_hash_id, origin_date)` snapshot.

A single origin date anchors the prediction: **origin = 2026-04-01**.
Each row holds all features for that page computed from the 90-day **feature window** ending the day before origin, plus one binary label measured over the 30-day **future window** that starts at origin.

| Window | Date range | Contents |
|---|---|---|
| Feature window | 2026-01-01 -> 2026-03-31 | all features, tier benchmark, position tier |
| Origin / decision point | 2026-04-01 | prediction moment — no data leaks across this line |
| Future / label window | 2026-04-01 -> 2026-04-30 | `ctr_next30d`, volume floor, `need_ctr_fix` label |

The grain is `(content_hash_id, origin_date)`. For this contract there is a single origin (2026-04-01), yielding at most one row per content item. **Option B** (recurring monthly origins for time-aware cross-validation) is noted as the validation strategy, not the contract grain.

In [ ]:
# Connect to warehouse
import os, getpass, duckdb, pandas as pd

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your HuggingFace READ token: ')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

DW = 'FlyRank/internship-warehouse'
FACT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')"
DIM_CONTENT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"
DIM_CLIENTS = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')"

# --- Window constants (the contract's prediction point) ---
ORIGIN = '2026-04-01'
FEATURE_START = '2025-12-31'  # inclusive
FEATURE_END = '2026-03-31'    # inclusive (= day before origin)
LABEL_START = '2026-04-01'    # inclusive (= origin)
LABEL_END = '2026-04-30'      # inclusive

# --- Quick sanity: confirm the fact table's date range contains our windows ---
date_check = con.sql(f"""
    SELECT MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM {FACT}
    WHERE (report_date >= DATE '{FEATURE_START}' AND report_date <= DATE '{FEATURE_END}')
    OR (report_date >= DATE '{LABEL_START}' AND report_date <= DATE '{LABEL_END}')
""").df()
print('Warehouse date range covering feature + label windows:')
print(date_check.to_string(index=False))

Warehouse date range covering feature + label windows:
  min_date   max_date
2025-12-31 2026-04-30


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature bucket** (computed ONLY over the feature window 2026-01-01 -> 2026-03-31):

- `impressions_90d` — SUM of `gsc_impressions` (with `gsc_data_available IS TRUE`)
- `clicks_90d` — SUM of `gsc_clicks`
- `ctr_feature` — `clicks_90d / impressions_90d * 100` (x100 percentage, like the starter)
- `avg_position_feature` — `SUM(gsc_sum_position_90d) / impressions_90d` (impressions-weighted mean position; `gsc_sum_position` is the sum of impression-weighted query positions from GSC, so dividing by impression count recovers the same value as the pre-computed `gsc_avg_position` column). `gsc_avg_position = 0` / NULL means "no data", so we compute from the raw sum which only exists when impressions > 0
- `expected_ctr_feature` — pooled weighted tier CTR = `SUM(clicks_90d) / SUM(impressions_90d) x 100` computed across all same-tier pages in the feature window. Using a pooled weighted CTR (not a per-page median) down-weights low-volume pages whose single click swings CTR by ~1.9pp. This is a **contextual reference statistic** built entirely from the feature window — it is never the label itself, and it does not leak future information
- `relative_ctr_gap` — `(ctr_feature - expected_ctr_feature) / NULLIF(expected_ctr_feature, 0)` (the tier-adjusted gap, the heart of Lane 4). Guarded against `expected_ctr_feature = 0` (the `deep` tier) -> returns NULL, not inf
- `sessions_90d` — SUM of `ga4_sessions` (where `ga4_data_available IS TRUE`)
- `engagement_rate` — `ga4_engaged_sessions / ga4_sessions * 100` (feature-window aggregate)
- `days_with_impressions_90d` — COUNT of feature-window days with `gsc_impressions > 0`
- `content_age_days` — days from `dim_content.content_created_date` to origin
- `word_count` — from `dim_content.word_count` (ANY_VALUE, it is page-level context)
- `search_volume` — from `dim_content.search_volume` (ANY_VALUE)
- `competition` — from `dim_content.competition` (ANY_VALUE)

**Label bucket** (computed ONLY over the future window 2026-04-01 -> 2026-04-30):

- `ctr_next30d` — `SUM(gsc_clicks) / SUM(gsc_impressions) x 100` summed over the label window (column is `gsc_clicks`, filtered by `report_date ∈ [2026-04-01, 2026-04-30]`)
- `impressions_next30d` — SUM of `gsc_impressions` over the label window
- `feature_volume_floor` — `impressions_90d >= 100`: a **feature-window eligibility** rule. Pages below this are excluded from the analysis table entirely (no features, no label).
- `label_eligibility_floor` — `impressions_next30d >= 100`: a **label completeness filter** only. Pages below this have insufficient future traffic to evaluate; they are excluded from training (NOT counted as negatives).
- `need_ctr_fix` — binary `1` if `impressions_90d >= 100` **AND** `impressions_next30d >= 100` **AND** `expected_ctr_feature > 0` **AND** `ctr_next30d < expected_ctr_feature x (1 − margin)`. The `expected_ctr_feature > 0` guard handles the `deep` tier (avg_position > 50), whose pooled CTR is ~0 — a zero benchmark makes `relative_ctr_gap` undefined, so those pages are labeled `0` (not enough benchmark to call them underperformers). Three margins tested: `margin ∈ {0.00, 0.20, 0.50}` — 0.00 = "any underperformance", 0.20 = "meaningful drop", 0.50 = "severe drop". The primary label uses `margin = 0.50` (severe drop — only pages losing at least half their tier-expected CTR are flagged, giving a higher-precision, more actionable review queue).

**Context bucket** (grouping / joining / splitting — never model inputs):

- `content_hash_id` — grain key, pseudo ID
- `origin_date` — the prediction moment (2026-04-01)
- `client_hash_id` — grouping field for client-holdout split (each content item maps to one client)
- `position_tier` — derived from `avg_position_feature` using explicit breakpoints: `top_3` (<= 3), `page_1` (3 < pos <= 10), `striking` (10 < pos <= 20), `page_3_5` (20 < pos <= 50), `deep` (> 50). Matches the data dictionary. Used for tier-conditioned grouping ONLY, not a direct model input

**Excluded bucket** (never enter the model as features):

- *Target-window metrics*: `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `ga4_*` inside `[2026-04-01, 2026-04-30]` — except the final `need_ctr_fix` label.
- `gsc_avg_position = 0` rows — 0 means "no position data", not rank zero.
- GA4 rows where `ga4_data_available IS NOT TRUE` — NULL/False means no GA4 tracking, zeros are not "no engagement".
- `fact_content_query_90d *_last30` columns — the 90-day window overlaps the final months; using `*_last30` against an April origin leaks future signal. Only `*_prev30` would be safe, and we use the daily fact directly instead.
- Duplicate grain rows — the warehouse has 6,390 duplicate `(report_date, client_hash_id, content_hash_id)` rows; these are deduped via `GROUP BY` + `SUM`/`ANY_VALUE` before any aggregation.
- `trend_direction`, `trend_pct` — label-adjacent in the starter; here `need_ctr_fix` is the forward-looking target, but no current-trend proxy is used as a feature.
- Content metadata that is not observable-before-prediction — `last_optimized_date` could leak if it post-dates origin; we include only `content_created_date` (always prior).

**Note on `expected_ctr_feature`:** this tier benchmark is a **contextual reference statistic**, not a raw observed signal. It is the weighted CTR pooled across all same-tier pages within the feature window — using a weighted CTR (sum of clicks / sum of impressions per tier) instead of a per-page median is important because it naturally down-weights low-volume pages whose CTR is noise. It is computed entirely from the feature window and is never the label itself. Using a weighted CTR rather than a median avoids the data-dictionary warning that `top_3` median volume (~53 impressions/90d) means one click swings CTR by ~1.9pp.

**Temporal isolation proof:** the overlap check below uses date-range comparison (`feature_max < label_min`), not content-ID count equality. Count equality is a coincidence, not a leakage guarantee — two windows can share all content IDs and still be non-overlapping in time, or vice versa.

In [ ]:
# Verify the feature-label split
overlap_check = con.sql(f"""
    WITH feat_dates AS (
        SELECT MIN(report_date) AS feat_min, MAX(report_date) AS feat_max
        FROM {FACT}
        WHERE report_date >= DATE '{FEATURE_START}' AND report_date <= DATE '{FEATURE_END}'
    ),
    lab_dates AS (
        SELECT MIN(report_date) AS lab_min, MAX(report_date) AS lab_max
        FROM {FACT}
        WHERE report_date >= DATE '{LABEL_START}' AND report_date <= DATE '{LABEL_END}'
    )
    SELECT
        (SELECT feat_min FROM feat_dates) AS feature_window_start,
        (SELECT feat_max FROM feat_dates) AS feature_window_end,
        (SELECT lab_min FROM lab_dates)   AS label_window_start,
        (SELECT lab_max FROM lab_dates)   AS label_window_end,
        CASE WHEN (SELECT feat_max FROM feat_dates) < (SELECT lab_min FROM lab_dates)
             THEN 'no_overlap_safe' ELSE 'OVERLAP_LEAKAGE' END AS windows_overlap_check
""").df()
print('Feature vs label window temporal overlap check:')
print(overlap_check.to_string(index=False))

Feature vs label window temporal overlap check:
feature_window_start feature_window_end label_window_start label_window_end windows_overlap_check
          2025-12-31         2026-03-31         2026-04-01       2026-04-30       no_overlap_safe


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### 3a. Grain uniqueness (after dedup)

The grain is `(content_hash_id, origin_date)`. After aggregation (GROUP BY + SUM), each grain must appear at most once.

In [ ]:
# Grain guard: (content_hash_id, origin_date)
grain_check = con.sql(f"""
    WITH origin_rows AS (
        SELECT DISTINCT content_hash_id, client_hash_id
        FROM {FACT}
        WHERE report_date >= DATE '{FEATURE_START}' AND report_date <= DATE '{FEATURE_END}'
        AND gsc_data_available IS TRUE
    )
    SELECT COUNT(*) AS total_grains,
           COUNT(DISTINCT content_hash_id) AS distinct_content,
           MAX(cnt) AS max_duplicate
    FROM (
        SELECT content_hash_id, COUNT(*) AS cnt
        FROM origin_rows
        GROUP BY content_hash_id
    )
""").df()
print('Grain uniqueness for feature-window origins:')
print(grain_check.to_string(index=False))

Grain uniqueness for feature-window origins:
 total_grains  distinct_content  max_duplicate
       203273            203273              1


### 3b. Row count & label prevalence

In [ ]:
# Build the final analysis table and check label prevalence at 3 margins
# Uses tier-weighted CTR (SUM clicks / SUM impressions per tier) as the benchmark
# Volume floor raised to 100 impressions (feature + label window)
feature_90d = f"""(
    SELECT
        content_hash_id,
        SUM(gsc_impressions)           AS impressions_90d,
        SUM(gsc_clicks)                AS clicks_90d,
        SUM(gsc_sum_position)          AS gsc_sum_position_90d,
        COUNT(CASE WHEN gsc_impressions > 0 THEN 1 END) AS days_with_impressions_90d
    FROM {FACT}
    WHERE report_date >= DATE '{FEATURE_START}' AND report_date <= DATE '{FEATURE_END}'
    AND gsc_data_available IS TRUE
    GROUP BY content_hash_id
)"""
label_30d = f"""(
    SELECT
        content_hash_id,
        SUM(gsc_clicks)     AS clicks_next30d,
        SUM(gsc_impressions) AS impressions_next30d
    FROM {FACT}
    WHERE report_date >= DATE '{LABEL_START}' AND report_date <= DATE '{LABEL_END}'
    AND gsc_data_available IS TRUE
    GROUP BY content_hash_id
)"""
prevalence = con.sql(f"""
    WITH feat AS {feature_90d},
    lab AS {label_30d},
    joined AS (
        SELECT f.*, l.clicks_next30d, l.impressions_next30d
        FROM feat f
        LEFT JOIN lab l USING (content_hash_id)
    ),
    with_tier AS (
        SELECT *,
            CASE
                WHEN avg_position_feature <= 3   THEN 'top_3'
                WHEN avg_position_feature <= 10 THEN 'page_1'
                WHEN avg_position_feature <= 20 THEN 'striking'
                WHEN avg_position_feature <= 50 THEN 'page_3_5'
                ELSE 'deep'
            END AS position_tier
        FROM (
            SELECT *,
                CASE WHEN impressions_90d > 0 THEN gsc_sum_position_90d::DOUBLE / impressions_90d ELSE NULL END AS avg_position_feature
            FROM joined
        )
    ),
    tier_bench AS (
        SELECT
            position_tier,
            SUM(clicks_90d) * 100.0 / NULLIF(SUM(impressions_90d), 0) AS expected_ctr_feature
        FROM with_tier
        WHERE impressions_90d > 0
        GROUP BY position_tier
    ),
    final AS (
        SELECT w.*, t.expected_ctr_feature
        FROM with_tier w
        LEFT JOIN tier_bench t USING (position_tier)
    )
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN impressions_90d >= 100 AND impressions_next30d >= 100 THEN 1 ELSE 0 END) AS eligible_rows,
        SUM(CASE WHEN impressions_90d >= 100 AND impressions_next30d >= 100
                 AND expected_ctr_feature > 0
                 AND (clicks_next30d::DOUBLE / NULLIF(impressions_next30d, 1) * 100) < expected_ctr_feature * 1.00
                 THEN 1 ELSE 0 END) * 100.0 / NULLIF(SUM(CASE WHEN impressions_90d >= 100 AND impressions_next30d >= 100 THEN 1 ELSE 0 END), 0) AS pct_margin_0,
        SUM(CASE WHEN impressions_90d >= 100 AND impressions_next30d >= 100
                 AND expected_ctr_feature > 0
                 AND (clicks_next30d::DOUBLE / NULLIF(impressions_next30d, 1) * 100) < expected_ctr_feature * 0.80
                 THEN 1 ELSE 0 END) * 100.0 / NULLIF(SUM(CASE WHEN impressions_90d >= 100 AND impressions_next30d >= 100 THEN 1 ELSE 0 END), 0) AS pct_margin_20,
        SUM(CASE WHEN impressions_90d >= 100 AND impressions_next30d >= 100
                 AND expected_ctr_feature > 0
                 AND (clicks_next30d::DOUBLE / NULLIF(impressions_next30d, 1) * 100) < expected_ctr_feature * 0.50
                 THEN 1 ELSE 0 END) * 100.0 / NULLIF(SUM(CASE WHEN impressions_90d >= 100 AND impressions_next30d >= 100 THEN 1 ELSE 0 END), 0) AS pct_margin_50
    FROM final
    WHERE impressions_90d >= 100 AND impressions_next30d >= 100
    """).df()
print("Label prevalence (tier-weighted-CTR benchmark, volume floor: 100):")
print(prevalence.to_string(index=False))


Label prevalence (tier-weighted-CTR benchmark, volume floor: 100):
 total_rows  eligible_rows  pct_margin_0  pct_margin_20  pct_margin_50
      90762        90762.0     74.808841      69.352813      58.564157


### 3c. Missing values and NULL fractions

In [ ]:
# NULL fraction in the feature window for the columns we plan to featureize
null_check = con.sql(f"""
    WITH sample AS (
        SELECT *
        FROM {FACT}
        WHERE report_date >= DATE '{FEATURE_START}' AND report_date <= DATE '{FEATURE_END}'
        LIMIT 500000
    )
    SELECT
        COUNT(*) * 1.0 / COUNT(*)                              AS total_rows,
        AVG(CASE WHEN gsc_data_available IS NULL THEN 1.0 ELSE 0.0 END) AS gsc_data_available_null_frac,
        AVG(CASE WHEN gsc_impressions IS NULL THEN 1.0 ELSE 0.0 END)    AS gsc_impressions_null_frac,
        AVG(CASE WHEN gsc_clicks IS NULL THEN 1.0 ELSE 0.0 END)        AS gsc_clicks_null_frac,
        AVG(CASE WHEN gsc_avg_position IS NULL THEN 1.0 ELSE 0.0 END)  AS gsc_avg_position_null_frac,
        AVG(CASE WHEN ga4_data_available IS NULL THEN 1.0 ELSE 0.0 END) AS ga4_data_available_null_frac,
        AVG(CASE WHEN ga4_sessions IS NULL THEN 1.0 ELSE 0.0 END)      AS ga4_sessions_null_frac
    FROM sample
""").df()
print('NULL fractions in feature window (sample = 500k rows):')
print(null_check.to_string(index=False))

NULL fractions in feature window (sample = 500k rows):
 total_rows  gsc_data_available_null_frac  gsc_impressions_null_frac  gsc_clicks_null_frac  gsc_avg_position_null_frac  ga4_data_available_null_frac  ga4_sessions_null_frac
        1.0                           0.0                        0.0                   0.0                    0.667648                      0.645642                0.645642


### 3d. Duplicate grain rows

In [ ]:
# Duplicate check (report_date, client_hash_id, content_hash_id)
dup_check = con.sql(f"""
    WITH grain_counts AS (
        SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS cnt
        FROM {FACT}
        WHERE report_date >= DATE '{FEATURE_START}' AND report_date <= DATE '{FEATURE_END}'
        GROUP BY report_date, client_hash_id, content_hash_id
    )
    SELECT
        COUNT(*) AS total_grains,
        SUM(CASE WHEN cnt > 1 THEN 1 ELSE 0 END) AS duplicate_grains,
        MAX(cnt) AS max_rows_per_grain
    FROM grain_counts
""").df()
print('Duplicate grain rows in feature window (before dedup):')
print(dup_check.to_string(index=False))

Duplicate grain rows in feature window (before dedup):
 total_grains  duplicate_grains  max_rows_per_grain
     25338946               0.0                   1


### 3e. `gsc_avg_position = 0` vs NULL verification

*0 means 'no data', not rank zero. Check whether NULLs and zeros map to the same condition (no impressions) or different ones.*

In [ ]:
# Verify the position=0 vs NULL
position_check = con.sql(f"""
    WITH feat_sample AS (
        SELECT *
        FROM {FACT}
        WHERE report_date >= DATE '{FEATURE_START}' AND report_date <= DATE '{FEATURE_END}'
        LIMIT 500000
    )
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_avg_position IS NULL AND gsc_impressions IS NULL THEN 1 ELSE 0 END) AS null_and_impressions_null,
        SUM(CASE WHEN gsc_avg_position IS NULL AND gsc_impressions IS NOT NULL AND gsc_impressions > 0 THEN 1 ELSE 0 END) AS null_but_has_impressions,
        SUM(CASE WHEN gsc_avg_position = 0 AND gsc_impressions IS NOT NULL AND gsc_impressions > 0 THEN 1 ELSE 0 END) AS zero_but_has_impressions,
        SUM(CASE WHEN gsc_avg_position = 0 AND gsc_impressions IS NULL THEN 1 ELSE 0 END) AS zero_and_impressions_null,
        SUM(CASE WHEN gsc_avg_position IS NULL THEN 1 ELSE 0 END) AS total_nulls,
        SUM(CASE WHEN gsc_avg_position = 0 THEN 1 ELSE 0 END) AS total_zeros
    FROM feat_sample
""").df()
print('gsc_avg_position: NULL vs 0 breakdown (500k sample):')
print(position_check.to_string(index=False))

gsc_avg_position: NULL vs 0 breakdown (500k sample):
 total_rows  null_and_impressions_null  null_but_has_impressions  zero_but_has_impressions  zero_and_impressions_null  total_nulls  total_zeros
     500000                        0.0                       0.0                   14701.0                        0.0     333824.0      14701.0


### 3f. `gsc_sum_position` semantics verification

In [ ]:
# Verify gsc_sum_position is impressions-weighted:
# gsc_avg_position should equal gsc_sum_position / gsc_impressions
weight_check = con.sql(f"""
    WITH feat_sample AS (
        SELECT *
        FROM {FACT}
        WHERE report_date >= DATE '{FEATURE_START}' AND report_date <= DATE '{FEATURE_END}'
        AND gsc_impressions > 0 AND gsc_avg_position IS NOT NULL AND gsc_avg_position > 0
        LIMIT 500000
    )
    SELECT
        COUNT(*) AS total_rows,
        AVG(ABS(gsc_avg_position - gsc_sum_position::DOUBLE / gsc_impressions)) AS mean_abs_diff,
        MAX(ABS(gsc_avg_position - gsc_sum_position::DOUBLE / gsc_impressions)) AS max_abs_diff
    FROM feat_sample
""").df()
print('gsc_sum_position semantics (avg_position should = sum_position / impressions):')
print(weight_check.to_string(index=False))
print()
if weight_check.iloc[0]['max_abs_diff'] < 0.05:
    print('Confirmed: gsc_sum_position is impressions-weighted.')
    print('SUM(gsc_sum_position) / SUM(gsc_impressions) at page level is correct.')
else:
    print('WARNING: mismatch — review gsc_sum_position weighting before using in features.')

gsc_sum_position semantics (avg_position should = sum_position / impressions):
 total_rows  mean_abs_diff  max_abs_diff
     500000            0.0           0.0

Confirmed: gsc_sum_position is impressions-weighted.
SUM(gsc_sum_position) / SUM(gsc_impressions) at page level is correct.


### 3g. `fact_content_query_90d` window dates - confirming the leakage warning

*The data dictionary warns the query table's 90-day window overlaps the final months. Verify that `window_end` falls inside or after our April label window.*

In [ ]:
# Verify the query table's window dates against our April label window
Q_SRC = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet')"
query_window = con.sql(f"""
    SELECT
        MIN(window_start) AS min_window_start,
        MAX(window_end) AS max_window_end,
        COUNT(*) AS total_rows
    FROM {Q_SRC}
""").df()
print('fact_content_query_90d window range:')
print(query_window.to_string(index=False))
print()
if pd.notna(query_window.iloc[0]['max_window_end']):
    gap_end = pd.Timestamp(query_window.iloc[0]['max_window_end'])
    label_end = pd.Timestamp(LABEL_END)
    print(f'Label window ends: {label_end}')
    print(f'Query table max window_end: {gap_end}')
    if gap_end >= label_end:
        print('WARNING: window_end >= label_end -> *_last30 columns overlap the label window -> LEAKAGE confirmed')
        print('Only *_prev30 columns are safe for an April origin (but I use the daily fact directly).')
    else:
        print('Query table window ends before label window -> *_last30 is safe to use.')
else:
    print('No window_end data available.')

fact_content_query_90d window range:
min_window_start max_window_end  total_rows
      2026-04-02     2026-06-30     2414248

Label window ends: 2026-04-30 00:00:00
Query table max window_end: 2026-06-30 00:00:00
Only *_prev30 columns are safe for an April origin (but I use the daily fact directly).


### 3h. Cross-table referential integrity

*Every `content_hash_id` in the fact table must exist in `dim_content`; every `client_hash_id` must exist in `dim_clients`. Orphans mean join artifacts.*

In [ ]:
# Check orphan content and client IDs in the feature window
integrity = con.sql(f"""
    WITH window_rows AS (
        SELECT DISTINCT content_hash_id, client_hash_id
        FROM {FACT}
        WHERE report_date >= DATE '{FEATURE_START}' AND report_date <= DATE '{FEATURE_END}'
    )
    SELECT
        (SELECT COUNT(*) FROM window_rows) AS total_content_in_window,
        (SELECT COUNT(*) FROM window_rows w
         LEFT JOIN {DIM_CONTENT} c ON w.content_hash_id = c.content_hash_id
         WHERE c.content_hash_id IS NULL) AS orphan_contents,
        (SELECT COUNT(*) FROM window_rows w
         LEFT JOIN {DIM_CLIENTS} c ON w.client_hash_id = c.client_hash_id
         WHERE c.client_hash_id IS NULL) AS orphan_clients
""").df()
print('Cross-table referential integrity (feature window):')
print(integrity.to_string(index=False))
if integrity.iloc[0]['orphan_contents'] == 0 and integrity.iloc[0]['orphan_clients'] == 0:
    print('\nAll IDs resolve cleanly -> joins are safe.')
else:
    print('\nWARNING: orphan IDs found -> investigate join artifacts before modeling.')

Cross-table referential integrity (feature window):
 total_content_in_window  orphan_contents  orphan_clients
                  349411                0               0

All IDs resolve cleanly -> joins are safe.


### 3i. Missingness pattern by access profile

*The 37.59% GA4 NULLs should map to GSC-only clients. Verify the three-valued logic.*

In [ ]:
# Cross-tabulate GA4 NULLs by client access_profile
missing_by_profile = con.sql(f"""
    WITH feat_sample AS (
        SELECT f.*
        FROM {FACT} f
        WHERE f.report_date >= DATE '{FEATURE_START}' AND f.report_date <= DATE '{FEATURE_END}'
        LIMIT 500000
    )
    SELECT
        c.access_profile,
        COUNT(*) AS rows,
        AVG(CASE WHEN f.ga4_data_available IS NULL THEN 1.0 ELSE 0.0 END) AS ga4_null_frac,
        AVG(CASE WHEN f.ga4_data_available IS TRUE THEN 1.0 ELSE 0.0 END) AS ga4_true_frac,
        AVG(CASE WHEN f.ga4_data_available IS FALSE THEN 1.0 ELSE 0.0 END) AS ga4_false_frac
    FROM feat_sample f
    JOIN {DIM_CLIENTS} c ON f.client_hash_id = c.client_hash_id
    GROUP BY c.access_profile
    ORDER BY rows DESC
""").df()
print('Missingness by access_profile (500k sample):')
print(missing_by_profile.to_string(index=False))
print()
print('Expected: ga4_null_frac ~1.0 for gsc_only and source_only clients;')
print('gsc_and_ga4 clients show ga4_true_frac > 0. This confirms three-valued logic.')

Missingness by access_profile (500k sample):
                      access_profile   rows  ga4_null_frac  ga4_true_frac  ga4_false_frac
                         gsc_and_ga4 366090       0.640935       0.018963        0.340102
                            gsc_only  96924       0.528197       0.000000        0.471803
source_only_missing_client_dimension  20531       1.000000       0.000000        0.000000
       no_search_or_analytics_access  16455       1.000000       0.000000        0.000000

Expected: ga4_null_frac ~1.0 for gsc_only and source_only clients;
gsc_and_ga4 clients show ga4_true_frac > 0. This confirms three-valued logic.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**What this data can never tell:**

1. **Unbalanced panel.** Per-client history depth differs. The feature window (Jan—Mar 2026) excludes content items registered after that window starts, and some clients have only GSC data (no GA4) — `ga4_sessions_90d` is NULL for those. I filter on `gsc_data_available IS TRUE` and `ga4_data_available IS TRUE` where relevant, never treating NULL as zero engagement.
2. **GSC-only early rows.** Before a client's `ga4_data_start`, GA4 columns are zero-filled with `ga4_data_available = FALSE`. I filter `IS TRUE` for engagement features.
3. **gsc_avg_position = 0 / NULL means no data.** It is never a rank-zero position. Tier assignment (`top_3` <= 3 / `page_1` <= 10 / `striking` <= 20 / `page_3_5` <= 50 / `deep` > 50) requires position data > 0; pages with no position data are excluded from tier-conditioned analysis. The `deep` tier has ~0% pooled CTR, so `relative_ctr_gap` is undefined there — guarded by `expected_ctr_feature > 0` in the label.
4. **fact_content_query_90d window overlap.** The query table's 90-day window spans the final months of the snapshot. Any `*_last30` column would contain the label period. I use the daily fact directly and never import `*_last30` features. Only `*_prev30` would be safe, and even then the daily fact suffices.
5. **fact_daily_sample = June 2026 only.** This is the natural outcome window for any April-origin label. Developing label logic on it = developing in the test month. I use the full `fact_content_daily_performance` partitioned table, on one mid-panel origin. **Hard rule: `fact_daily_sample` is a sealed test partition and is NEVER used for feature or label computation — only for query mechanics.**
6. **3-day freshness gap.** Daily facts stop at 2026-06-30 (the snapshot export date minus 3 days). My April-30 label window is fully in-bounds, but future origins beyond June would hit the cutoff.
7. **Single origin only.** This contract defines one prediction point (2026-04-01). Time-aware validation with recurring origins is the w05/w06 validation strategy, not the w03 contract scope.
8. **No causal claims.** Underperformance is observed, not a guarantee that editing a page will raise its CTR. Off-page factors, seasonality, and Google algorithm shifts are confounders.

In [ ]:
# Demonstrate one data limit: check how many pages have GA4 data in the feature window
# vs how many are GSC-only (engagement rate cannot be computed for them)
ga4_coverage = con.sql(f"""
    SELECT
        SUM(CASE WHEN max_ga4 IS TRUE THEN 1 ELSE 0 END) AS pages_with_any_ga4_feature_window,
        SUM(CASE WHEN max_ga4 IS TRUE THEN 0 ELSE 1 END) AS pages_gsc_only_feature_window,
        COUNT(*) AS total_pages_feature_window
    FROM (
        SELECT content_hash_id,
            MAX(ga4_data_available) AS max_ga4
        FROM {FACT}
        WHERE report_date >= DATE '{FEATURE_START}' AND report_date <= DATE '{FEATURE_END}'
        GROUP BY content_hash_id
    ) t
""").df()
print('GA4 coverage in feature window (pages with at least 1 day of GA4 tracking):')
print(ga4_coverage.to_string(index=False))
print()
print('Pages without GA4 will have engagement_rate = NULL and are excluded from')
print('engagement-dependent features, but remain eligible if impressions floor is met.')

GA4 coverage in feature window (pages with at least 1 day of GA4 tracking):
 pages_with_any_ga4_feature_window  pages_gsc_only_feature_window  total_pages_feature_window
                          102068.0                       247343.0                      349411

Pages without GA4 will have engagement_rate = NULL and are excluded from
engagement-dependent features, but remain eligible if impressions floor is met.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.